# Save and load

`app.save(path)` writes the graph as a versioned JSON file and `app.load(path)`
reads it back, into any app that knows the same node types. This notebook saves a
small graph, looks at the file, loads it into a second app and checks that both
give the same result. Then it feeds `load` files that are not graphs and shows
the error each one gets.

In [1]:
# Standard library
import json
import math
from pathlib import Path

In [2]:
# Local libraries
import element_51

In [3]:
# Files go in a scratch folder, never in the project
OUT = Path("__data__")
OUT.mkdir(exist_ok=True)
GRAPH = OUT / "graph.json"
DOUBLED = 8.0

## Save a graph

In [4]:
@element_51.node
def number() -> float:
    return 4.0


@element_51.node
def double(value: float) -> float:
    return value * 2


app = element_51.App([number, double])
first = app.add_node(number, x=60.0, y=100.0)
second = app.add_node(double, x=320.0, y=100.0)
app.add_link(first, "out", second, "value")
app.save(GRAPH)
print(GRAPH, GRAPH.stat().st_size, "bytes")

__data__/graph.json 445 bytes


The file is plain JSON, with a `format_version` so later versions can read it.

In [5]:
saved = json.loads(GRAPH.read_text("utf-8"))
print(json.dumps(saved, indent=4)[:600])
assert "format_version" in saved
assert len(saved["nodes"]) == len(app.snapshot().graph.nodes)

{
    "format_version": 2,
    "nodes": [
        {
            "id": "n1",
            "node_type": "number",
            "x": 60.0,
            "y": 100.0,
            "values": {}
        },
        {
            "id": "n2",
            "node_type": "double",
            "x": 320.0,
            "y": 100.0,
            "values": {}
        }
    ],
    "links": [
        {
            "source": {
                "node": "n1",
                "port": "out"
            },
            "target": {
                "node": "n2",
                "port": "value"
            }
        }
    ]
}


## Load it into a fresh app

In [6]:
copy = element_51.App([number, double])
copy.load(GRAPH)
before = app.run()
after = copy.run()
print(before)
print(after)
assert before == after
assert math.isclose(after[second]["out"], DOUBLED)

{'n1': {'out': 4.0}, 'n2': {'out': 8.0}}
{'n1': {'out': 4.0}, 'n2': {'out': 8.0}}


## Files that are not graphs

`load` raises `StoreError` and says what is wrong with the file, so a person can
tell a truncated file from a file in the wrong format.

In [7]:
def try_load(name: str, content: bytes) -> str:
    path = OUT / name
    path.write_bytes(content)
    try:
        element_51.App([number, double]).load(path)
    except element_51.StoreError as error:
        return str(error)
    return "loaded"


cases = {
    "not_json.json": b"this is not json",
    "not_utf8.json": b"\xff\xfe\x00",
    "wrong_shape.json": b"[1, 2, 3]",
}
messages = {name: try_load(name, content) for name, content in cases.items()}
for name, message in messages.items():
    print(f"{name}: {message}")
assert "loaded" not in messages.values()

not_json.json: __data__/not_json.json is not valid JSON: Expecting value: line 1 column 1 (char 0)
not_utf8.json: __data__/not_utf8.json is not a graph file: it is not UTF-8 text
wrong_shape.json: __data__/wrong_shape.json is not a graph file: 1 validation error for _Header
  Input should be a valid dictionary or instance of _Header [type=model_type, input_value=[1, 2, 3], input_type=list]
    For further information visit https://errors.pydantic.dev/2.14/v/model_type
